# Sinhala VITS TTS — Google Colab Guide

Train a **multi-speaker VITS** model with Coqui TTS on cleaned OpenSLR Resource 30 Sinhala data.

## Before you start

1. **Runtime → Change runtime type → GPU** (T4 is fine).
2. Upload prepared dataset from your PC to Drive:
   `MyDrive/sinhala-vits/dataset/`
   containing `wavs/`, `metadata_train.csv`, `metadata_val.csv`.
3. Run cells **in order**.
4. After the install cell, do **Runtime → Restart session**, then continue from the **Setup paths** cell.

## Data note

Official transcripts have fewer rows than cleaned WAVs. Training uses matched IDs only (~1248).

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

## 2) Install packages + clone repo

Uses maintained `coqui-tts` (not old `TTS`) and `transformers>=4.57`.

**After this cell finishes successfully:**
`Runtime → Restart session`

Then continue from section 3.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/model-using-openSLR")
REPO_URL = "https://github.com/DSEgrp18/model-using-openSLR.git"

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

# Remove abandoned packages that break on Colab Python 3.12
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "TTS", "trainer"],
    check=False,
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "coqui-tts[notebooks]",
        "transformers>=4.57.0",
        "librosa",
        "soundfile",
        "tqdm",
    ],
    check=True,
)

print("Install finished.")
print("NOW DO: Runtime → Restart session")
print("Then run the next Setup paths cell.")

## 3) Setup paths (run this after every runtime restart)

Mount Drive and redefine folders. Safe to re-run anytime.

In [ ]:
import os
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/sinhala-vits")
DATASET_DIR = DRIVE_ROOT / "dataset"
OUTPUT_DIR = DRIVE_ROOT / "outputs"
RAW_AUDIO_DIR = DRIVE_ROOT / "dataset_clean"
TRANSCRIPT = DRIVE_ROOT / "si_lk.lines.txt"
REPO = Path("/content/model-using-openSLR")

assert REPO.exists(), "Repo missing. Re-run the Install cell first."
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import TTS
import transformers
from trainer import Trainer  # noqa: F401

print("Repo:", REPO)
print("Dataset:", DATASET_DIR)
print("Outputs:", OUTPUT_DIR)
print("TTS:", getattr(TTS, "__version__", "unknown"))
print("transformers:", transformers.__version__)
print("Setup OK")

## 4) Prepare dataset (skip if already uploaded prepared folder)

If `MyDrive/sinhala-vits/dataset/metadata_train.csv` already exists, this cell only verifies it.

Otherwise it builds the dataset from:
- `MyDrive/sinhala-vits/dataset_clean/`
- `MyDrive/sinhala-vits/si_lk.lines.txt`

In [ ]:
import subprocess
import sys

need_prepare = not (DATASET_DIR / "metadata_train.csv").exists()
if need_prepare:
    assert RAW_AUDIO_DIR.exists(), f"Upload cleaned WAVs to {RAW_AUDIO_DIR}"
    assert TRANSCRIPT.exists(), f"Upload transcript to {TRANSCRIPT}"
    subprocess.run(
        [
            sys.executable,
            "scripts/prepare_vits_dataset.py",
            "--audio-dir", str(RAW_AUDIO_DIR),
            "--transcript-file", str(TRANSCRIPT),
            "--output-dir", str(DATASET_DIR),
            "--sample-rate", "22050",
            "--seed", "42",
        ],
        check=True,
    )
else:
    print("Prepared dataset already found at", DATASET_DIR)

train_n = sum(1 for _ in (DATASET_DIR / "metadata_train.csv").open(encoding="utf-8"))
val_n = sum(1 for _ in (DATASET_DIR / "metadata_val.csv").open(encoding="utf-8"))
wav_n = len(list((DATASET_DIR / "wavs").glob("*.wav")))
print(f"train lines={train_n}, val lines={val_n}, wavs={wav_n}")

## 5) Dataset smoke check

In [ ]:
from collections import Counter

train_rows = (DATASET_DIR / "metadata_train.csv").read_text(encoding="utf-8").splitlines()
val_rows = (DATASET_DIR / "metadata_val.csv").read_text(encoding="utf-8").splitlines()
speakers = Counter(line.split("|")[2] for line in train_rows if line.strip())
missing = [
    line.split("|")[0]
    for line in train_rows + val_rows
    if line.strip() and not (DATASET_DIR / "wavs" / f"{line.split('|')[0]}.wav").exists()
]

print(f"train={len(train_rows)} val={len(val_rows)} speakers={len(speakers)}")
print("speakers:", dict(speakers))
assert not missing, f"Missing wavs example: {missing[:10]}"
assert len(speakers) >= 1, "No speakers found in metadata"
print("Dataset check passed")

## 6) Train VITS

| GPU | batch_size |
|-----|------------|
| T4 16GB | 8 |
| L4 24GB | 12–16 |
| A100 | 24–32 |

If Colab disconnects later, set `CONTINUE_PATH` to the run folder under `outputs/` and re-run this cell.

In [ ]:
import subprocess
import sys

BATCH_SIZE = 8           # use 8 on T4; raise if you have more VRAM
EPOCHS = 1000
CONTINUE_PATH = ""       # example: "/content/drive/MyDrive/sinhala-vits/outputs/sinhala_vits_openslr30-September-..."

cmd = [
    sys.executable, "scripts/train_vits.py",
    "--dataset-path", str(DATASET_DIR),
    "--output-path", str(OUTPUT_DIR),
    "--run-name", "sinhala_vits_openslr30",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", "4",
    "--num-loader-workers", "2",
    "--sample-rate", "22050",
]
if CONTINUE_PATH:
    cmd += ["--continue-path", CONTINUE_PATH]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## 7) Pick checkpoint

In [ ]:
runs = sorted(OUTPUT_DIR.glob("sinhala_vits_openslr30*"))
assert runs, f"No runs found in {OUTPUT_DIR}"
RUN_DIR = runs[-1]
print("Using run:", RUN_DIR)

checkpoints = sorted(RUN_DIR.glob("*.pth"))
for path in checkpoints[-10:]:
    print(path.name, path.stat().st_size // (1024 * 1024), "MB")

CONFIG_PATH = RUN_DIR / "config.json"
best = list(RUN_DIR.glob("best_model*.pth"))
MODEL_PATH = best[-1] if best else (checkpoints[-1] if checkpoints else None)
assert MODEL_PATH is not None and CONFIG_PATH.exists(), "Checkpoint/config missing"
print("MODEL:", MODEL_PATH)
print("CONFIG:", CONFIG_PATH)

## 8) Synthesize Sinhala speech

In [ ]:
from IPython.display import Audio, display
from TTS.utils.synthesizer import Synthesizer

TEXT = "ආයුබෝවන්, මම සිංහලෙන් කතා කරනවා."
SPEAKER = "sin_2241"  # examples: sin_2241, sin_4191, sin_6314
OUT_WAV = OUTPUT_DIR / "samples" / f"{SPEAKER}_demo.wav"
OUT_WAV.parent.mkdir(parents=True, exist_ok=True)

synth = Synthesizer(
    tts_checkpoint=str(MODEL_PATH),
    tts_config_path=str(CONFIG_PATH),
    use_cuda=True,
)
wav = synth.tts(text=TEXT, speaker_name=SPEAKER)
synth.save_wav(wav, str(OUT_WAV))
print("Wrote", OUT_WAV)
display(Audio(str(OUT_WAV)))

## Troubleshooting

- **OOM:** set `BATCH_SIZE = 4` or `8`, restart, continue from Setup paths.
- **`isin_mps_friendly` error:** re-run Install cell, restart runtime, then Setup paths.
- **Disconnected:** remount Drive via Setup paths, set `CONTINUE_PATH`, re-run Train.
- **Missing dataset:** upload `data/prepared/sinhala_vits` to `MyDrive/sinhala-vits/dataset/`.

OpenSLR Resource 30 is CC BY-SA 4.0 (Google).